# CZI image basics: demo and tests

This notebook runs each function in the `czi_basics` package on the sample file in `data/`
and checks the results with `assert` statements. An `assert` does nothing when its
condition is true and stops with an error when it is false, so **if every cell runs
without an error, the code works.**

The sample, `data/sample_nuclei_foci.czi`, is a small crop of a real image of
irradiated cells with two channels:

* channel 0, **DAPI**: stains DNA, so it shows the nuclei
* channel 1, **GFP**: shows the DNA damage repair foci (the small bright dots)

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

# our custom package (install it first with: pip install .)
from czi_basics import load_czi, save_image, to_8bit

SAMPLE = Path("data/sample_nuclei_foci.czi")
OUT = Path("outputs")   # converted images are written here

## 1. `load_czi`: read the file

We get back a NumPy array with one 2D image per channel, plus metadata.

In [ ]:
data, meta = load_czi(SAMPLE)
meta

In [ ]:
# check that the file was read correctly
assert data.shape == (2, 512, 512)              # 2 channels, 512 x 512 pixels
assert data.dtype == np.uint16                  # raw microscope data is 16-bit
assert meta["channel_names"] == ["DAPI", "GFP"]
assert abs(meta["pixel_size_um"]["X"] - 0.1095) < 1e-3   # ~0.11 µm per pixel

dapi, gfp = data   # split the array into its two channels
print("load_czi OK")
print("GFP values range from", gfp.min(), "to", gfp.max(), "(out of 65535 possible)")

## 2. `to_8bit`: 16-bit to 8-bit grayscale

Shown raw, the 16-bit data looks dark and flat because the signal uses only a
small part of the 0-65535 range. Stretching the contrast to 0-255 fixes that.

In [ ]:
dapi8 = to_8bit(dapi)
gfp8 = to_8bit(gfp, low_pct=1, high_pct=99.9)

# the result is 8-bit and uses the full black-to-white range
for img in (dapi8, gfp8):
    assert img.dtype == np.uint8
    assert img.min() == 0 and img.max() == 255

# edge cases: a flat image, and a tiny example we can check by hand
assert (to_8bit(np.full((4, 4), 7)) == 0).all()
assert to_8bit(np.array([[0, 50], [100, 100]]), 0, 100).tolist() == [[0, 128], [255, 255]]
print("to_8bit OK")

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
axes[0].imshow(gfp, cmap="gray", vmin=0, vmax=65535); axes[0].set_title("GFP raw, full 16-bit range")
axes[1].imshow(dapi8, cmap="gray"); axes[1].set_title("DAPI, 8-bit")
axes[2].imshow(gfp8, cmap="gray");  axes[2].set_title("GFP foci, 8-bit")
for ax in axes:
    ax.axis("off")
plt.show()

## 3. `save_image`: write PNG and TIFF files

In [ ]:
written = [
    save_image(dapi8, OUT / "sample_DAPI.png"),
    save_image(gfp8, OUT / "sample_GFP.png"),
    save_image(gfp, OUT / "sample_GFP_16bit.tif"),   # TIFF can keep the raw 16-bit values
]

for p in written:
    assert p.exists()
    print(p, Image.open(p).size, Image.open(p).mode)

# read the files back: both formats must give exactly the pixels we saved
assert (np.array(Image.open(OUT / "sample_GFP.png")) == gfp8).all()
assert (np.array(Image.open(OUT / "sample_GFP_16bit.tif")) == gfp).all()

# a colour (3D) array is rejected with a clear message
try:
    save_image(np.zeros((10, 10, 3), dtype=np.uint8), OUT / "colour.png")
except ValueError as err:
    print("expected error:", err)

print("save_image OK")

## Putting it together

The whole conversion is a few lines, so it is easy to loop over a folder of
`.czi` files.

In [ ]:
def czi_to_pngs(czi_path, out_dir):
    """Convert every channel of one .czi file into an 8-bit PNG."""
    data, meta = load_czi(czi_path)
    saved = []
    for name, channel in zip(meta["channel_names"], data):
        out_path = Path(out_dir) / f"{Path(czi_path).stem}_{name}.png"
        saved.append(save_image(to_8bit(channel), out_path))
    return saved

czi_to_pngs(SAMPLE, OUT)

In [ ]:
# the usual way to show this kind of image: nuclei in blue, foci in green
overlay = np.zeros((512, 512, 3), dtype=np.uint8)
overlay[..., 2] = dapi8   # blue channel
overlay[..., 1] = gfp8    # green channel

plt.figure(figsize=(5, 5))
plt.imshow(overlay)
plt.axis("off")
plt.title("DAPI (blue) + GFP foci (green)")
plt.show()
print("All checks passed.")